# Modul 8 · Klassifikation und Modellbewertung
**Data Science 1** · linkit courses

Die Instandhaltung will mechanische Ausfälle eine Schicht vorher erkennen. Du baust die Zielgröße selbst nach, schätzt logistische Regressionen mit statsmodels und scikit-learn, teilst die Daten nach der Zeit, erzeugst Leckage mit Absicht, wählst eine Schwelle nach Kosten, prüfst die Kalibrierung und fragst, für welche Maschinen das Modell taugt.

**Lernziele**
1. Eine logistische Regression schätzen und als Odds-Verhältnisse deuten
2. Daten zeitlich teilen, Leckage erkennen und Pipelines richtig aufbauen
3. Konfusionsmatrix, Precision, Recall und F1 berechnen und deuten
4. Schwelle nach Kosten wählen, ROC- und Precision-Recall-Kurve lesen
5. Kalibrierung von Wahrscheinlichkeiten prüfen und beurteilen
6. Regularisierung mit Kreuzvalidierung wählen und gegen Vergleichsmodelle messen
7. Modellgüte für Teilgruppen prüfen und Verschiebungen im Betrieb erkennen

**So arbeitest du**
- Das Notebook läuft auf der Kursseite direkt im Browser, ohne Installation. Alternativ öffnest du es in Google Colab und speicherst dort zuerst eine eigene Kopie (*Datei → Kopie in Drive speichern*).
- Führe die Zellen von oben nach unten mit **Shift + Enter** aus. Der erste Import von statsmodels lädt im Browser SciPy und statsmodels, der erste von scikit-learn zusätzlich scikit-learn. Beides dauert einmalig etwa 10 Sekunden.
- Stellen mit `# TODO` füllst du selbst aus, meist mit ein bis zehn Zeilen. Darunter prüft ein Selbsttest dein Ergebnis. Ein ❌ ist nur ein Hinweis, dass etwas noch nicht stimmt, das Notebook läuft trotzdem weiter.
- Zellen, die ein Ergebnis aus einem TODO brauchen, melden „⏳ Erst TODO n lösen.“, solange es fehlt.
- Mindestziel sind TODO 1 bis 4, TODO 6 und 7 sowie TODO 9 bis 13. TODO 5, TODO 8 und die Bonusaufgaben vertiefen.
- TODO-Zellen mit Zufall legen den Seed fest, etwa mit `KFold(..., random_state=8)`. Die Selbsttests lassen so viel Spielraum, dass jeder richtige Weg besteht.
- Gib Ergebnisse mit `float(...)`, `round(...)` oder der Hilfsfunktion `zeige` aus, dann steht im Browser und in Colab dasselbe da. Jede Grafikzelle endet mit `plt.show()`.
- Jeder Abschnitt gehört zu einer Lerneinheit und zum Abschnitt mit derselben Nummer im Skript. Hilfen zu jedem TODO und typische Fehler findest du auf der Kursseite.

In [ ]:
# Hilfsfunktion für die Selbsttests: einfach ausführen, nicht ändern.
import math

def check(name, test):
    """Führt einen Test aus und zeigt ✅ oder ❌. Fehler im Test gelten als „noch nicht gelöst“."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

print("Bereit.")

In [ ]:
# Daten laden: Auf der Kursseite liegen die Dateien schon bereit, in Colab holt diese Zelle sie von GitHub.
import os, urllib.request
DATEN = ["sensordaten.csv", "maschinen.csv", "stoerungen.csv", "produktion.csv"]
QUELLE = "https://raw.githubusercontent.com/juliankunz18/linkit-courses/main/data-science-1/daten/"
os.makedirs("daten", exist_ok=True)
for datei in DATEN:
    if not os.path.exists("daten/" + datei):
        urllib.request.urlretrieve(QUELLE + datei, "daten/" + datei)
print("Daten bereit:", ", ".join(sorted(os.listdir("daten"))))

## 1 · Von der Geraden zur Wahrscheinlichkeit
*Lerneinheit 1 · Skript Abschnitt 1*

Die **logistische Regression** setzt den Logit, den Logarithmus der Odds, linear an:

$$\ln\frac{p}{1 - p} = \beta_0 + \beta_1 x_1 + \dots + \beta_k x_k, \qquad p = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \dots)}}.$$

Geschätzt wird mit Maximum Likelihood. Steigt $x_1$ um eins, werden die Odds mit dem **Odds-Verhältnis** $e^{\beta_1}$ multipliziert, die Wahrscheinlichkeit selbst steigt je nach Ausgangslage verschieden stark.

Die nächste Zelle lädt die Daten wie im Skript: `daten` hat eine Zeile je Maschine und Produktionsschicht mit Schwingstärke `vibration_mm_s`, Lagertemperatur `lagertemp_c`, Stromaufnahme `strom_pct`, Linie, Maschinentyp und der Zielgröße `ausfall_naechste`. `jahr1` ist das erste Jahr bis zum 30. September 2025 zum Trainieren, `jahr2` das zweite zum Prüfen. Dazu kommen `stoerungen` und `produktion` für TODO 2. Die Hilfsfunktion `zeige` druckt Zahlen mit 6 gültigen Stellen, Tabellen mit ihrer Größe und lange Arrays mit Länge und Summe.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

SCHICHT_TYP = pd.CategoricalDtype(["Früh", "Spät", "Nacht"], ordered=True)
sensoren = pd.read_csv("daten/sensordaten.csv", dtype={"schicht": SCHICHT_TYP})
maschinen = pd.read_csv("daten/maschinen.csv")
stoerungen = pd.read_csv("daten/stoerungen.csv", dtype={"schicht": SCHICHT_TYP})
produktion = pd.read_csv("daten/produktion.csv", dtype={"schicht": SCHICHT_TYP})
daten = sensoren.merge(maschinen[["maschine", "linie", "typ"]], on="maschine", validate="many_to_one")
jahr1 = daten[daten["datum"] < "2025-10-01"]
jahr2 = daten[daten["datum"] >= "2025-10-01"]
print(daten.shape, int(daten["ausfall_naechste"].sum()), round(daten["ausfall_naechste"].mean(), 4))


def zeige(*werte):
    """Druckt Zahlen mit 6 gültigen Stellen, damit im Browser und in Colab dieselben Ziffern stehen.
    Tabellen erscheinen mit ihrer Größe, lange Arrays mit Länge und Summe. None bleibt None."""
    def text(w):
        if w is None or isinstance(w, (str, bool)):
            return str(w)
        if isinstance(w, dict):
            return "{" + ", ".join(f"{k}: {text(v)}" for k, v in w.items()) + "}"
        if isinstance(w, pd.DataFrame):
            return f"Tabelle {w.shape[0]} × {w.shape[1]}"
        if np.ndim(w) == 0:
            return f"{float(w):.6g}"
        w = np.ravel(w)
        if len(w) > 12:
            return f"[{len(w)} Werte, Summe {float(np.sum(w)):.6g}]"
        return "[" + " ".join(f"{float(v):.6g}" for v in w) + "]"
    print(*[text(w) for w in werte])

Die Zerspanungsmaschinen im ersten Jahr wie im Skript: Ausfallwahrscheinlichkeit gegen Schwingstärke mit `smf.logit`, Odds-Verhältnis mit Intervall und Wahrscheinlichkeiten bei 3 bis 6 mm/s.

In [ ]:
zersp = jahr1[jahr1["typ"] == "Zerspanung"]
m_zer = smf.logit("ausfall_naechste ~ vibration_mm_s", data=zersp).fit(disp=0)
ki = m_zer.conf_int()
tabelle = pd.DataFrame({"b": m_zer.params, "se": m_zer.bse, "OR": np.exp(m_zer.params), "OR_unten": np.exp(ki[0]), "OR_oben": np.exp(ki[1])})
print(len(zersp), int(m_zer.nobs), round(zersp["ausfall_naechste"].mean(), 4))
print(tabelle.round(4))
neu = pd.DataFrame({"vibration_mm_s": [3.0, 4.0, 5.0, 6.0]})
p_neu = m_zer.predict(neu).to_numpy()
odds_neu = p_neu / (1 - p_neu)
print(p_neu.round(4), (odds_neu[1:] / odds_neu[:-1]).round(4), (p_neu[1:] / p_neu[:-1]).round(4))

### TODO 1 · Logistische Regression für Linie 3
Linie 3 fertigt Schweißbaugruppen. Sagt die Schwingstärke dort Ausfälle genauso gut voraus? `linie3` enthält die Schichten von Linie 3 im ersten Jahr.
1. `m_l3`: das angepasste Modell `smf.logit("ausfall_naechste ~ vibration_mm_s", data=linie3).fit(disp=0)`.
2. `or_l3`: das Odds-Verhältnis je mm/s als Zahl, `or_l3_ki`: sein 95-%-Intervall als Array mit zwei Werten.
3. `p_l3`: die vorhergesagten Ausfallwahrscheinlichkeiten bei 2 und bei 5 mm/s als Array. `rr_l3`: wievielmal so hoch die Wahrscheinlichkeit bei 5 wie bei 2 mm/s ist. `or_3mm`: das Odds-Verhältnis für 3 mm/s mehr.

In [ ]:
linie3 = jahr1[jahr1["linie"] == 3]

# TODO 1a
m_l3 = None

# TODO 1b
or_l3 = None
or_l3_ki = None

# TODO 1c
p_l3 = None
rr_l3 = None
or_3mm = None

zeige(or_l3, or_l3_ki, p_l3, rr_l3, or_3mm)

In [ ]:
check("TODO 1a Modell für Linie 3", lambda: int(m_l3.nobs) == 2862 and np.allclose(m_l3.params.to_numpy(), [-4.0148916, 0.30372701], atol=1e-6))
check("TODO 1b Odds-Verhältnis mit Intervall", lambda: math.isclose(or_l3, 1.35489914, abs_tol=1e-6)
      and np.allclose(or_l3_ki, [1.21666328, 1.50884119], atol=1e-6))
check("TODO 1c Wahrscheinlichkeiten und Verhältnisse", lambda: np.allclose(p_l3, [0.03206383, 0.07612103], atol=1e-7)
      and math.isclose(rr_l3, 2.37404672, abs_tol=1e-6) and math.isclose(or_3mm, 2.48725836, abs_tol=1e-6))

In Linie 3 werden die Odds je mm/s nur mit 1,35 multipliziert, in der Zerspanung mit 3,46. Von 2 auf 5 mm/s steigt die Ausfallwahrscheinlichkeit von 3,2 % auf 7,6 %, auf das 2,37-fache, die Odds steigen auf das 2,49-fache. Bei so seltenen Ereignissen liegen beide Verhältnisse nah beieinander, bei höheren Wahrscheinlichkeiten nicht mehr. Das Intervall von 1,22 bis 1,51 schließt die 1 aus: Auch in Linie 3 sagt die Schwingung etwas voraus, nur weniger.

## 2 · Trainieren, validieren, testen
*Lerneinheit 2 · Skript Abschnitt 2*

Trainiert wird mit der Vergangenheit, geprüft mit der Zukunft. **Leckage** heißt, dass ins Training Information gelangt, die beim Einsatz noch nicht bekannt wäre: durch einen Verschiebungsfehler beim Bauen von Zielgröße oder Merkmalen, durch Vorverarbeitung auf allen Daten oder durch eine zufällige statt einer zeitlichen Teilung. Eine **Pipeline** bündelt Vorverarbeitung und Modell, `fit` passt jeden Schritt nur mit den Daten an, die es bekommt.

### TODO 2 · Die Zielgröße nachbauen
`ausfall_naechste` ist 1, wenn die Maschine in der **nächsten Produktionsschicht** mechanische Störungen mit zusammen mindestens 60 Minuten Stillstand hat. Die nächste Produktionsschicht steht im Arbeitskalender: nach einer Freitagsnachtschicht die Frühschicht am Samstag, wenn dort gearbeitet wird, sonst am Montag, nach Feiertagen und Werksferien entsprechend später. `produktion` enthält jede Produktionsschicht, die Spalte `schicht` ist nach Früh, Spät und Nacht geordnet.
1. `kalender`: eine Tabelle aller Produktionsschichten mit den Spalten `datum` und `schicht`, zeitlich sortiert, dazu `naechstes_datum` und `naechste_schicht` für die jeweils folgende Produktionsschicht.
2. `mech_min`: der mechanische Stillstand je Tag, Schicht und Maschine aus `stoerungen`, als Tabelle mit den Spalten `datum`, `schicht`, `maschine` und `stillstand_min`.
3. `ziel`: für jede Zeile von `daten` wahr, wenn die Maschine in der nächsten Produktionsschicht mindestens 60 Minuten mechanischen Stillstand hat, als Array. `uebereinstimmung`: der Anteil der Zeilen, in denen `ziel` mit `ausfall_naechste` übereinstimmt.

In [ ]:
# TODO 2a
kalender = None

# TODO 2b
mech_min = None
ziel = None
uebereinstimmung = None

zeige(kalender, uebereinstimmung, ziel)

In [ ]:
def naechste(datum, schicht):
    zeile = kalender[(kalender["datum"] == datum) & (kalender["schicht"] == schicht)].iloc[0]
    return zeile["naechstes_datum"], zeile["naechste_schicht"]

check("TODO 2a Arbeitskalender", lambda: len(kalender) == 1447 and naechste("2024-10-04", "Nacht") == ("2024-10-07", "Früh")
      and naechste("2024-12-23", "Nacht") == ("2025-01-02", "Früh") and naechste("2024-10-08", "Nacht") == ("2024-10-09", "Früh"))
check("TODO 2b Zielgröße wie ausfall_naechste", lambda: len(ziel) == 28920 and int(np.sum(ziel)) == 1599 and uebereinstimmung == 1.0)

Die nachgebaute Zielgröße stimmt in allen 28.920 Zeilen. Auf den Freitag, 4. Oktober 2024, folgt der Montag, auf den 23. Dezember 2024 der 2. Januar 2025. Die letzte Schicht des Kalenders hat keine Nachfolgerin, deshalb fehlt sie in `sensordaten.csv`. Wer den Stillstand der aktuellen statt der nächsten Schicht nimmt, trifft dieselbe Zahl von 1.599 Ausfällen, aber nur in 89,8 % der Zeilen denselben Wert: Die Verschiebung ist leicht zu übersehen.

Das Hauptmodell wie im Skript: drei Messgrößen mit Median-Ersatz und Standardisierung, der Maschinentyp als Dummies, logistische Regression. `p1` und `p2` sind die Ausfallwahrscheinlichkeiten im ersten und zweiten Jahr.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score

MESS = ["vibration_mm_s", "lagertemp_c", "strom_pct"]
MERKMALE = MESS + ["typ"]
vorverarbeitung = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), MESS),
    (OneHotEncoder(handle_unknown="ignore"), ["typ"]))
modell = make_pipeline(vorverarbeitung, LogisticRegression(solver="newton-cholesky"))
modell.fit(jahr1[MERKMALE], jahr1["ausfall_naechste"])
p1 = modell.predict_proba(jahr1[MERKMALE])[:, 1]
p2 = modell.predict_proba(jahr2[MERKMALE])[:, 1]
y1, y2 = jahr1["ausfall_naechste"].to_numpy(), jahr2["ausfall_naechste"].to_numpy()
print(round(roc_auc_score(y1, p1), 4), round(roc_auc_score(y2, p2), 4), round(p1.mean(), 4), round(y1.mean(), 4), round(p2.mean(), 4), round(y2.mean(), 4))

### TODO 3 · Eine Pipeline mit der Maschine statt dem Typ
Jede Maschine hat ihre eigene Ausfallquote. Nimm statt des Typs die Maschine als Kategorie.
1. `modell_m`: eine Pipeline wie das Hauptmodell, aber mit `OneHotEncoder(handle_unknown="ignore")` für die Spalte `maschine`, angepasst mit dem ersten Jahr. Die Spalten stehen in `MERKMALE_M`.
2. `p2_m`: die Ausfallwahrscheinlichkeiten im zweiten Jahr, `auc_m`: die AUC im zweiten Jahr. `mittel_m`: die Mittelwerte, die der `StandardScaler` in der Pipeline gelernt hat. Du erreichst ihn mit `modell_m[0].named_transformers_["pipeline"][-1]`.

In [ ]:
MERKMALE_M = MESS + ["maschine"]

# TODO 3a
modell_m = None

# TODO 3b
p2_m = None
auc_m = None
mittel_m = None

zeige(auc_m, mittel_m)

In [ ]:
check("TODO 3a Pipeline mit der Maschine", lambda: type(modell_m).__name__ == "Pipeline" and modell_m[-1].coef_.shape == (1, 23))
check("TODO 3b Mittelwerte nur aus dem ersten Jahr", lambda: np.allclose(mittel_m, [2.46145607, 32.56595537, 88.05647141], atol=1e-6))
check("TODO 3b AUC im zweiten Jahr", lambda: len(p2_m) == 14580 and math.isclose(auc_m, 0.73641154, abs_tol=1e-6))

In [ ]:
if auc_m is None:
    print("⏳ Erst TODO 3 lösen.")
else:
    print(f"AUC mit dem Typ {roc_auc_score(y2, p2):.4f}, mit der Maschine {auc_m:.4f}")
    print(f"Mittelwerte aus allen Daten wären {daten[MESS].mean().round(3).tolist()}")

Mit der Maschine statt dem Typ kommen 23 statt 13 Koeffizienten heraus, die AUC sinkt im zweiten Jahr leicht von 0,7402 auf 0,7364. Mehr Kategorien helfen nicht automatisch. Der `StandardScaler` hat die Mittelwerte des ersten Jahres gelernt, 2,461 mm/s für die Schwingstärke. Aus allen Daten wären es 2,485 mm/s, darin steckt der Sensortausch des zweiten Jahres.

### TODO 4 · Leckage selbst erzeugen
Auch die Zahl der Störungsmeldungen einer Schicht ist ein naheliegendes Merkmal. Baue es einmal richtig und einmal mit dem Verschiebungsfehler.
1. `mit_meld`: eine Kopie von `daten` mit der Spalte `meldungen`, der Zahl der Meldungen aller Kategorien für die Maschine in derselben Schicht (0, wenn es keine gab), und der Spalte `meldungen_naechste` mit der Zahl in der nächsten Produktionsschicht derselben Maschine. Tipp: `groupby("maschine")["meldungen"].shift(-1)`.
2. `auc_meld` und `auc_leck`: die AUC im zweiten Jahr für das Hauptmodell mit `meldungen` beziehungsweise mit `meldungen_naechste` als vierter Messgröße, trainiert mit dem ersten Jahr. Die Zeilen des ersten Jahres findest du mit `mit_meld.loc[jahr1.index]`.

In [ ]:
# TODO 4a
mit_meld = None

# TODO 4b
auc_meld = None
auc_leck = None

zeige(mit_meld, auc_meld, auc_leck)

In [ ]:
check("TODO 4a Meldungen je Schicht", lambda: len(mit_meld) == 28920 and int(mit_meld["meldungen"].sum()) == 14256
      and int(mit_meld["meldungen_naechste"].sum()) == 14243 and int(mit_meld["meldungen_naechste"].isna().sum()) == 20)
check("TODO 4b AUC richtig und mit Leckage", lambda: math.isclose(auc_meld, 0.7402851, abs_tol=1e-6) and math.isclose(auc_leck, 0.8987798, abs_tol=1e-6))

Mit den Meldungen der aktuellen Schicht bleibt die AUC bei 0,740, mit denen der nächsten steigt sie auf 0,899. Wer meldet, hat eine Störung, und die nächste Schicht enthält die Ausfälle, die vorhergesagt werden sollen. Von den 14.263 Meldungen fehlen 7 in `mit_meld`, sie stammen aus der letzten Schicht, die keine Zeile hat.

Die Kreuzvalidierung wie im Skript, mit fünf Falten:

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit, KFold, cross_val_score

for cv in [TimeSeriesSplit(n_splits=5), KFold(n_splits=5, shuffle=True, random_state=1)]:
    auc = cross_val_score(clone(modell), jahr1[MERKMALE], y1, cv=cv, scoring="roc_auc")
    print(type(cv).__name__, auc.round(4), round(auc.mean(), 4), round(auc.std(), 4))

### TODO 5 · Zeitlich oder zufällig falten
Wiederhole den Vergleich mit vier Falten. `kfold` ist schon angelegt, mit festem Seed.
1. `auc_zeit`: die AUC je Falte mit `TimeSeriesSplit(n_splits=4)` auf dem ersten Jahr, für eine ungelernte Kopie des Hauptmodells.
2. `auc_kfold`: dasselbe mit `kfold`.

In [ ]:
kfold = KFold(n_splits=4, shuffle=True, random_state=8)

# TODO 5
auc_zeit = None
auc_kfold = None

zeige(auc_zeit, auc_kfold)

In [ ]:
check("TODO 5 zeitliche Falten", lambda: np.allclose(auc_zeit, [0.70639097, 0.77701453, 0.76140812, 0.77448071], atol=1e-6))
check("TODO 5 zufällige Falten", lambda: len(auc_kfold) == 4 and 0.750 <= np.mean(auc_kfold) <= 0.762
      and not np.allclose(auc_kfold, [0.70639097, 0.77701453, 0.76140812, 0.77448071], atol=1e-3))

Mit der Zeit schwankt die AUC je Falte von 0,706 bis 0,777, im Mittel 0,755. Die zufälligen Falten liegen mit Seed 8 im Mittel bei 0,758 und streuen weniger, weil jede Falte Schichten aus dem ganzen Jahr enthält. Beide Schätzungen liegen über den 0,740 des zweiten Jahres: Was sich im nächsten Jahr ändert, kann keine Falte des ersten Jahres zeigen.

## 3 · Die Konfusionsmatrix
*Lerneinheit 3 · Skript Abschnitt 3*

Bei einer Schwelle gibt es für jede Schicht einen Alarm oder keinen. TP sind erkannte Ausfälle, FP Fehlalarme, FN verpasste Ausfälle, TN ruhige Schichten ohne Alarm:

$$\text{Precision} = \frac{TP}{TP + FP}, \quad \text{Recall} = \frac{TP}{TP + FN}, \quad F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}, \quad \text{Accuracy} = \frac{TP + TN}{n}.$$

### TODO 6 · Konfusionsmatrix bei 0,2 und das Accuracy-Paradox
Das Skript nimmt die Schwelle 0,1, hier ist es 0,2. `alarm_02` enthält die Alarme des Hauptmodells im zweiten Jahr. Die Zelle lädt auch die Kennzahlen aus scikit-learn, zur Probe etwa `precision_score(y2, alarm_02)`.
1. `tp6`, `fp6`, `fn6`, `tn6`: die vier Felder als ganze Zahlen, von Hand aus `alarm_02` und `y2` gezählt. `precision6`, `recall6`, `f1_6` und `accuracy6` mit den Formeln oben.
2. Linie 4 hat nur 3,6 % Ausfälle. `acc_nie_l4`: die Accuracy der Regel „nie Alarm“ in Linie 4. `acc_modell_l4` und `recall_modell_l4`: Accuracy und Recall des Modells bei 0,2 in Linie 4. Die Zeilen von Linie 4 stehen in `l4`.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

alarm_02 = p2 >= 0.2
l4 = (jahr2["linie"] == 4).to_numpy()

# TODO 6a
tp6 = None
fp6 = None
fn6 = None
tn6 = None
precision6 = None
recall6 = None
f1_6 = None
accuracy6 = None

# TODO 6b
acc_nie_l4 = None
acc_modell_l4 = None
recall_modell_l4 = None

zeige(tp6, fp6, fn6, tn6, precision6, recall6, f1_6, accuracy6, acc_nie_l4, acc_modell_l4, recall_modell_l4)

In [ ]:
check("TODO 6a vier Felder", lambda: (tp6, fp6, fn6, tn6) == (215, 377, 588, 13400) and confusion_matrix(y2, alarm_02).ravel().tolist() == [tn6, fp6, fn6, tp6])
check("TODO 6a Kennzahlen", lambda: math.isclose(precision6, 0.36317568, abs_tol=1e-7) and math.isclose(recall6, 0.26774595, abs_tol=1e-7)
      and math.isclose(f1_6, 0.30824373, abs_tol=1e-7) and math.isclose(accuracy6, 0.93381344, abs_tol=1e-7)
      and math.isclose(precision6, precision_score(y2, alarm_02)) and math.isclose(f1_6, f1_score(y2, alarm_02)))
check("TODO 6b Accuracy-Paradox in Linie 4", lambda: math.isclose(acc_nie_l4, 0.96410608, abs_tol=1e-7) and math.isclose(acc_modell_l4, 0.96319159, abs_tol=1e-7)
      and math.isclose(recall_modell_l4, 0.05095541, abs_tol=1e-7))

Bei 0,2 gibt es 592 Alarme, davon 215 echte Ausfälle: Precision 0,363, Recall 0,268, $F_1 = 0{,}308$. Gegenüber 0,1 aus dem Skript ist jeder dritte Alarm ein Treffer, dafür bleiben fast drei Viertel der Ausfälle unentdeckt. In Linie 4 liegt „nie Alarm“ zu 96,4 % richtig, das Modell mit 96,3 % sogar etwas seltener, obwohl es 5 % der Ausfälle findet. Die Accuracy belohnt das Nichtstun.

## 4 · Schwelle, ROC und Kosten
*Lerneinheit 4 · Skript Abschnitt 4*

Die **ROC-Kurve** trägt für jede Schwelle den Recall gegen die Falsch-positiv-Rate $FP/(FP + TN)$ auf, ihre Fläche ist die **AUC**: die Wahrscheinlichkeit, dass ein zufälliger Ausfall eine höhere Wahrscheinlichkeit bekommt als eine zufällige Schicht ohne, Gleichstände zählen halb. Die **Precision-Recall-Kurve** misst gegen die Basisrate, ihre Fläche schätzt die mittlere Precision AP. Bei Kosten $C_{FP}$ für einen Fehlalarm und $C_{FN}$ für die Mehrkosten eines verpassten Ausfalls lohnt sich ein Alarm ab $p^* = C_{FP}/(C_{FP} + C_{FN})$, sofern die Wahrscheinlichkeiten kalibriert sind.

### TODO 7 · ROC-Punkte von Hand für Linie 1
`l1_j2` enthält die Schichten von Linie 1 im zweiten Jahr, `y_l1` ihre Ausgänge.
1. `p_l1`: die Ausfallwahrscheinlichkeiten des Hauptmodells für diese Schichten. `roc_punkte`: ein Array mit vier Zeilen (Falsch-positiv-Rate, Recall) für die Schwellen 0,05, 0,1, 0,2 und 0,3, von Hand gezählt, Alarm bei $p \ge$ Schwelle.
2. `auc_rang`: die AUC als Anteil der Paare aus einem Ausfall und einer Schicht ohne, in denen der Ausfall die höhere Wahrscheinlichkeit hat, Gleichstände halb. Tipp: `p_l1[y_l1 == 1][:, None] - p_l1[y_l1 == 0][None, :]` bildet alle Differenzen. `auc_l1`: zum Vergleich mit `roc_auc_score`.

In [ ]:
l1_j2 = jahr2[jahr2["linie"] == 1]
y_l1 = l1_j2["ausfall_naechste"].to_numpy()

# TODO 7a
p_l1 = None
roc_punkte = None

# TODO 7b
auc_rang = None
auc_l1 = None

zeige(roc_punkte, auc_rang, auc_l1)

In [ ]:
check("TODO 7a ROC-Punkte von Hand", lambda: np.allclose(roc_punkte, [[0.42259414, 0.77926421], [0.15630604, 0.61204013], [0.0385535, 0.34782609], [0.01524208, 0.20735786]], atol=1e-7))
check("TODO 7b AUC als Rangwahrscheinlichkeit", lambda: math.isclose(auc_rang, 0.78256671, abs_tol=1e-7) and math.isclose(auc_l1, auc_rang, abs_tol=1e-12))

Bei 0,1 bekommt Linie 1 in 15,6 % der ruhigen Schichten einen Alarm und findet 61,2 % der Ausfälle. Von allen Paaren aus einem der 299 Ausfälle und einer der 3.346 ruhigen Schichten gewinnt der Ausfall in 78,3 %. Mit `modell.predict(...)` statt der Wahrscheinlichkeiten gäbe es nur 0 und 1 und eine AUC von 0,545.

### TODO 8 · Precision-Recall je Linie
1. `ap_linie` und `basis_linie`: Wörterbücher mit der mittleren Precision des Hauptmodells und der Ausfallquote je Linie im zweiten Jahr, Schlüssel 1 bis 4. `faktor_linie`: das Verhältnis beider je Linie.
2. Zeichne die Precision-Recall-Kurven von Linie 1 und Linie 4 mit `precision_recall_curve` in eine Grafik und die beiden Basisraten als waagrechte Linien.

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve

# TODO 8a
ap_linie = None
basis_linie = None
faktor_linie = None

# TODO 8b
fig, ax = plt.subplots(figsize=(7, 4))

zeige(ap_linie, faktor_linie)
plt.show()

In [ ]:
check("TODO 8a mittlere Precision je Linie", lambda: np.allclose([ap_linie[l_] for l_ in [1, 2, 3, 4]], [0.37568420, 0.30227151, 0.28631961, 0.09929052], atol=1e-7))
check("TODO 8a Verhältnis zur Basisrate", lambda: np.allclose([basis_linie[l_] for l_ in [1, 2, 3, 4]], [0.08203018, 0.05788752, 0.04663923, 0.03589392], atol=1e-7)
      and np.allclose([faktor_linie[l_] for l_ in [1, 2, 3, 4]], [4.57982916, 5.22170457, 6.13902923, 2.76622137], atol=1e-6))

Linie 1 hat mit 0,376 die höchste mittlere Precision, gemessen an der Basisrate liegt aber Linie 3 vorn: das 6,1-fache ihrer Ausfallquote von 4,7 %. In Linie 4 kommt das Modell nur auf das 2,8-fache, die Kurve fällt schnell auf die Basisrate.

### TODO 9 · Eine Kostenkurve mit anderen Annahmen
Angenommen, eine Inspektion kostet 500 € und eine Minute Stillstand 25 €. Die mittlere Dauer eines Ausfalls im ersten Jahr steht in `gross.mean()`, die Schwellen in `schwellen_k`. Beides legt die nächste Zelle an, die wie im Skript mit 300 € und 30 € je Minute rechnet.
1. `k_ausfall9`: die Kosten eines verpassten Ausfalls. `p_stern9`: die Schwelle nach der Faustformel.
2. `kosten9`: die Kosten im ersten Jahr für jede Schwelle aus `schwellen_k` als Array, 500 € je Alarm plus `k_ausfall9` je verpasstem Ausfall. `s_best9`: die Schwelle mit den kleinsten Kosten.
3. `kosten_test9`: ein Array mit den Kosten im zweiten Jahr bei `s_best9`, bei `p_stern9` und bei 0,5.

In [ ]:
mech = stoerungen[stoerungen["kategorie"] == "mechanisch"].groupby(["datum", "schicht", "maschine"], observed=True)["stillstand_min"].sum()
gross = mech[(mech >= 60) & (mech.index.get_level_values("datum") < "2025-10-01")]
K_INSP, K_MIN = 300, 30
k_ausfall = gross.mean() * K_MIN
p_stern = K_INSP / k_ausfall
print(len(gross), round(gross.mean(), 2), round(k_ausfall), round(p_stern, 4))
schwellen_k = np.round(np.arange(0.01, 0.51, 0.01), 2)
kosten1 = [K_INSP * (p1 >= s).sum() + k_ausfall * ((p1 < s) & (y1 == 1)).sum() for s in schwellen_k]
s_best = schwellen_k[np.argmin(kosten1)]
print(s_best)

In [ ]:
K_INSP9, K_MIN9 = 500, 25

# TODO 9a
k_ausfall9 = None
p_stern9 = None

# TODO 9b
kosten9 = None
s_best9 = None

# TODO 9c
kosten_test9 = None

zeige(k_ausfall9, p_stern9, s_best9, kosten_test9)

In [ ]:
check("TODO 9a Kosten und Faustformel", lambda: math.isclose(k_ausfall9, 2647.48427673, abs_tol=1e-6) and math.isclose(p_stern9, 0.18885853, abs_tol=1e-7))
check("TODO 9b kostenminimale Schwelle", lambda: len(kosten9) == 50 and math.isclose(kosten9[15], 1770749.37106918, abs_tol=1e-3) and math.isclose(s_best9, 0.16))
check("TODO 9c Kosten im zweiten Jahr", lambda: np.allclose(kosten_test9, [1849404.08805031, 1852245.91194969, 2018318.23899371], atol=1e-3))

Mit teureren Inspektionen und billigerem Stillstand verschiebt sich das Minimum von 0,13 auf 0,16, die Faustformel liegt bei 0,189. Im zweiten Jahr kostet 0,16 rund 1,849 Mio. €, die Faustformel 1,852 Mio. €, die Schwelle 0,5 2,018 Mio. €. Auf dem flachen Boden der Kurve kommt es auf die genaue Schwelle wenig an, auf die Größenordnung sehr. Welche Schwelle gilt, hängt an Annahmen, die das Werk festlegen muss.

## 5 · Wahrscheinlichkeiten, denen man trauen kann
*Lerneinheit 5 · Skript Abschnitt 5*

Ein Modell ist **kalibriert**, wenn die vorhergesagten Wahrscheinlichkeiten im Mittel den beobachteten Quoten entsprechen. `calibration_curve(y, p, n_bins=..., strategy="quantile")` bildet gleich große Klassen und gibt zuerst die beobachteten Quoten, dann die mittleren Vorhersagen zurück. Der **Brier-Score** $\frac1n\sum_i (p_i - y_i)^2$ und der **Log-Loss** bewerten Wahrscheinlichkeiten direkt, Vergleich ist die Basisrate des ersten Jahres. `class_weight="balanced"` gibt den Ausfällen so viel Gewicht wie allen anderen Schichten zusammen.

### TODO 10 · Kalibrierung des Maschinenmodells mit und ohne Gewichte
Nimm die Pipeline mit der Maschine aus TODO 3 und baue sie hier neu, damit die Zelle auch ohne TODO 3 läuft.
1. `p2_u10` und `p2_b10`: die Ausfallwahrscheinlichkeiten im zweiten Jahr ohne Gewichte und mit `class_weight="balanced"`, beide mit dem ersten Jahr angepasst.
2. `kal10`: eine Tabelle mit den Spalten `vorhergesagt`, `beobachtet_u`, `vorhergesagt_b` und `beobachtet_b` aus `calibration_curve` mit fünf gleich großen Klassen für beide Modelle.
3. `brier10`: ein Array mit den Brier-Scores ohne Gewichte, mit Gewichten und der Basisrate des ersten Jahres.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

# TODO 10a
p2_u10 = None
p2_b10 = None

# TODO 10b
kal10 = None

# TODO 10c
brier10 = None

zeige(kal10, brier10)

In [ ]:
check("TODO 10a Wahrscheinlichkeiten", lambda: math.isclose(float(np.mean(p2_u10)), 0.05709257, abs_tol=1e-7) and math.isclose(float(np.mean(p2_b10)), 0.40612152, abs_tol=1e-7))
check("TODO 10b Kalibrierungstabelle", lambda: np.allclose(kal10[["vorhergesagt", "beobachtet_u"]].to_numpy().T, [[0.01061415, 0.02258075, 0.03468396, 0.05496396, 0.16262001],
      [0.02263374, 0.02709191, 0.03360768, 0.03806584, 0.15397805]], atol=1e-7)
      and np.allclose(kal10[["vorhergesagt_b", "beobachtet_b"]].to_numpy().T, [[0.16345993, 0.29122245, 0.38160546, 0.48854782, 0.70577194],
      [0.02297668, 0.02640604, 0.03292181, 0.03840878, 0.15466392]], atol=1e-7))
check("TODO 10c Brier-Scores", lambda: np.allclose(brier10, [0.04627236, 0.19166369, 0.05204233], atol=1e-7))

Ohne Gewichte sagt das Maschinenmodell in der untersten Klasse 1,1 % voraus, beobachtet sind 2,3 %, in der obersten 16,3 % gegen 15,4 %. Mit Gewichten sagt es in der obersten Klasse 70,6 % voraus, beobachtet sind wieder nur 15,5 %. Die beobachteten Quoten je Klasse sind fast gleich, die Reihenfolge bleibt also erhalten. Der Brier-Score steigt aber von 0,0463 auf 0,1917 und ist damit fast viermal so schlecht wie die Basisrate mit 0,0520.

## 6 · Regularisierung und Vergleichsmodelle
*Lerneinheit 6 · Skript Abschnitt 6*

scikit-learn minimiert $\frac12\sum_j \beta_j^2 - C\,\ell(\beta)$: Kleines `C` bestraft große Koeffizienten stark. Welches `C` am besten vorhersagt, zeigt die Kreuzvalidierung auf dem ersten Jahr, nie das Testjahr. Ein Modell lohnt sich erst, wenn es Vergleichsmodelle und einfache Regeln schlägt.

### TODO 11 · C wählen und gegen eine Regel messen
1. `cv_mittel`: ein Wörterbuch mit der mittleren AUC über `TimeSeriesSplit(n_splits=4)` auf dem ersten Jahr für `C` = 0,001, 0,01, 0,1, 1 und 10, Schlüssel ist `C`. Nimm dazu jeweils `make_pipeline(clone(vorverarbeitung), LogisticRegression(solver="newton-cholesky", C=...))`. `bestes_c`: das `C` mit der größten mittleren AUC.
2. `auc_test_bestes`: die AUC im zweiten Jahr für dieses `C`, angepasst mit dem ganzen ersten Jahr.
3. Die Regel aus dem Skript nimmt die Schwingung. `regel_temp`: dieselbe Regel mit der Lagertemperatur, also deren Abstand zum Median der eigenen Maschine im ersten Jahr, fehlende Werte als 0. `auc_regel_temp`: ihre AUC im zweiten Jahr.

In [ ]:
# TODO 11a
cv_mittel = None
bestes_c = None

# TODO 11b
auc_test_bestes = None

# TODO 11c
regel_temp = None
auc_regel_temp = None

zeige(cv_mittel, bestes_c, auc_test_bestes, auc_regel_temp)

In [ ]:
check("TODO 11a Kreuzvalidierung je C", lambda: np.allclose([cv_mittel[c_] for c_ in [0.001, 0.01, 0.1, 1, 10]], [0.72008557, 0.72710212, 0.74654286, 0.75482358, 0.75561616], atol=1e-6)
      and bestes_c == 10)
check("TODO 11b AUC im zweiten Jahr", lambda: math.isclose(auc_test_bestes, 0.73992625, abs_tol=1e-6))
check("TODO 11c Regel mit der Lagertemperatur", lambda: len(regel_temp) == 14580 and math.isclose(auc_regel_temp, 0.69892468, abs_tol=1e-6))

Die Kreuzvalidierung wählt `C` = 10 mit 0,7556, knapp vor `C` = 1 mit 0,7548. Im zweiten Jahr erreicht `C` = 10 eine AUC von 0,7399, `C` = 1 hätte dort 0,7402. Wer `C` nach dem Testjahr wählt, nimmt 1 und hat das Testjahr damit schon benutzt, die 0,7402 wäre dann zu optimistisch. Die Regel mit der Lagertemperatur erreicht 0,699, etwas mehr als die mit der Schwingung im Skript, aber deutlich weniger als das Modell.

## 7 · Für wen funktioniert das Modell?
*Lerneinheit 7 · Skript Abschnitt 7*

Eine Gesamt-AUC mittelt über verschiedene Maschinen. Kalibrierung und Alarmquote je Gruppe zeigen, wo das Modell taugt und wo sich etwas verschoben hat. Am 2. März 2026 wurden die Schwingungssensoren an allen Maschinen der Linie 2 getauscht.

### TODO 12 · Güte je Linie und der Sensortausch je Maschine
1. `je_linie12`: eine Tabelle mit einer Zeile je Linie (Index 1 bis 4) und den Spalten `auc`, `vorhergesagt` (mittlere Wahrscheinlichkeit), `beobachtet` (Ausfallquote) und `alarme` (Anteil der Schichten mit $p \ge 0{,}2$), alles im zweiten Jahr.
2. `tausch12`: eine Tabelle mit einer Zeile je Maschine der Linie 2 (Index `M-01` bis `M-13`) und den Spalten `vorher` und `nachher`: der Anteil der Schichten mit $p \ge 0{,}1$ im zweiten Jahr vor und ab dem 2. März 2026.

In [ ]:
# TODO 12a
je_linie12 = None

# TODO 12b
tausch12 = None

zeige(je_linie12, tausch12)

In [ ]:
check("TODO 12a Güte je Linie", lambda: list(je_linie12.index) == [1, 2, 3, 4] and np.allclose(je_linie12["auc"], [0.78256671, 0.77449646, 0.70686627, 0.61245278], atol=1e-7))
check("TODO 12a Kalibrierung und Alarme je Linie", lambda: np.allclose(je_linie12["vorhergesagt"], [0.0743833, 0.07711291, 0.0378568, 0.03567762], atol=1e-7)
      and np.allclose(je_linie12["beobachtet"], [0.08203018, 0.05788752, 0.04663923, 0.03589392], atol=1e-7)
      and np.allclose(je_linie12["alarme"], [0.06392318, 0.08449931, 0.010631, 0.00457247], atol=1e-7))
check("TODO 12b Alarme je Maschine vor und nach dem Tausch", lambda: list(tausch12.index) == ["M-01", "M-02", "M-11", "M-12", "M-13"]
      and np.allclose(tausch12[["vorher", "nachher"]].to_numpy(), [[0.12052117, 0.45023697], [0.06188925, 0.47393365], [0.09120521, 0.28199052],
                                                                   [0.1465798, 0.42654028], [0.0, 0.00236967]], atol=1e-7))

Linie 2 hat mit 0,774 fast dieselbe AUC wie Linie 1, sagt aber im Mittel 7,7 % voraus bei 5,8 % beobachteten Ausfällen, Linie 1 dagegen 7,4 % bei 8,2 %. In Linie 3 und 4 gibt es bei 0,2 kaum noch Alarme. An den Fräszentren M-01 und M-02 steigt der Anteil der Alarme nach dem Tausch, im Mittel über März bis September, von 12 % und 6 % auf 45 % und 47 %. Die Koordinatenmessmaschine M-13 schwingt so wenig, dass sie auch mit den neuen Sensoren fast nie einen Alarm bekommt.

### TODO 13 · Ein Modellsteckbrief als Funktion
Schreib eine Funktion `steckbrief(p, y, schwelle, name)`, die für Wahrscheinlichkeiten `p`, Ausgänge `y` und eine Schwelle ein Wörterbuch mit den Schlüsseln `name`, `auc`, `ap`, `brier`, `basisrate`, `alarme`, `precision`, `recall` und `satz` zurückgibt. `alarme` ist die Zahl der Schichten mit $p \ge$ Schwelle als ganze Zahl. `satz` hat genau diese Form, mit Dezimalkomma und Tausenderpunkt:

`Hauptmodell: AUC 0,740, AP 0,273 bei Basisrate 0,055, Brier-Score 0,0461; bei der Schwelle 0,1 gibt es 1.825 Alarme, Precision 0,209, Recall 0,476.`

AUC, AP, Basisrate, Precision und Recall stehen mit drei Nachkommastellen, der Brier-Score mit vier.

In [ ]:
def steckbrief(p, y, schwelle, name):
    """Kennzahlen eines Modells bei einer Schwelle als Wörterbuch mit einem Satz für den Bericht."""
    # TODO 13
    return None


karte_haupt = steckbrief(p2, y2, 0.1, "Hauptmodell")
karte_streng = steckbrief(p2, y2, 0.3, "Hauptmodell")
print(None if karte_haupt is None else karte_haupt["satz"])

In [ ]:
check("TODO 13 Kennzahlen", lambda: math.isclose(karte_haupt["auc"], 0.74015195, abs_tol=1e-7) and math.isclose(karte_haupt["ap"], 0.27281611, abs_tol=1e-7)
      and math.isclose(karte_haupt["brier"], 0.04605331, abs_tol=1e-7) and math.isclose(karte_haupt["basisrate"], 0.05507545, abs_tol=1e-7)
      and karte_haupt["alarme"] == 1825 and math.isclose(karte_haupt["precision"], 0.20931507, abs_tol=1e-7) and math.isclose(karte_haupt["recall"], 0.47571606, abs_tol=1e-7))
check("TODO 13 Satz", lambda: karte_haupt["satz"] == "Hauptmodell: AUC 0,740, AP 0,273 bei Basisrate 0,055, Brier-Score 0,0461; bei der Schwelle 0,1 gibt es 1.825 Alarme, Precision 0,209, Recall 0,476.")
check("TODO 13 strengere Schwelle", lambda: karte_streng["alarme"] == 263 and math.isclose(karte_streng["precision"], 0.49429658, abs_tol=1e-7)
      and karte_streng["satz"].endswith("bei der Schwelle 0,3 gibt es 263 Alarme, Precision 0,494, Recall 0,162."))

Bei 0,3 gibt es nur 263 Alarme, fast jeder zweite trifft, gefunden werden aber nur 16,2 % der Ausfälle. Ein vollständiger Steckbrief nennt außerdem Zweck, Daten, Teilgruppen, Annahmen der Kostenrechnung und die Überwachung im Betrieb, wie im Skript.

### Bonus B1 · Rekalibrieren
Das gewichtete Hauptmodell gibt viel zu hohe Wahrscheinlichkeiten aus. `CalibratedClassifierCV` passt in jeder Falte das Modell an und auf der Validierung eine Umrechnung der Wahrscheinlichkeiten.
1. `kalibriert`: `CalibratedClassifierCV` mit dem gewichteten Hauptmodell (eine ungelernte Pipeline wie das Hauptmodell, aber `class_weight="balanced"`), `method="sigmoid"` (Platt-Skalierung) und `cv=TimeSeriesSplit(n_splits=3)`, angepasst mit dem ersten Jahr.
2. `p2_kal`: die Wahrscheinlichkeiten im zweiten Jahr. `brier_kal`: ihr Brier-Score, `mittel_kal`: ihr Mittelwert.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

# Bonus B1
kalibriert = None
p2_kal = None
brier_kal = None
mittel_kal = None

zeige(brier_kal, mittel_kal)

In [ ]:
check("B1 Brier-Score nach der Rekalibrierung", lambda: len(p2_kal) == 14580 and math.isclose(brier_kal, 0.04612341, abs_tol=1e-7))
check("B1 mittlere Wahrscheinlichkeit", lambda: math.isclose(mittel_kal, 0.05402110, abs_tol=1e-7))

Nach der Platt-Skalierung liegt die mittlere Wahrscheinlichkeit bei 5,4 % statt 40,7 %, der Brier-Score bei 0,0461 wie beim ungewichteten Modell. Die Gewichte haben die Reihenfolge kaum verändert, die Umrechnung holt die Höhe zurück. Einfacher ist es trotzdem, gar nicht erst zu gewichten.

### Bonus B2 · k-nächste Nachbarn
1. `auc_knn`: ein Wörterbuch mit der AUC im zweiten Jahr für `KNeighborsClassifier(n_neighbors=k)` mit $k$ = 10, 50 und 200, jeweils hinter `clone(vorverarbeitung)` in einer Pipeline, angepasst mit dem ersten Jahr.
2. `auc_knn_roh`: die AUC für $k$ = 200 ohne Standardisierung, nur mit Median-Ersatz für die Messgrößen und Dummies für den Typ.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Bonus B2
auc_knn = None
auc_knn_roh = None

zeige(auc_knn, auc_knn_roh)

In [ ]:
check("B2 AUC je k", lambda: np.allclose([auc_knn[k_] for k_ in [10, 50, 200]], [0.67316225, 0.70697657, 0.72494220], atol=1e-6))
check("B2 ohne Standardisierung", lambda: math.isclose(auc_knn_roh, 0.71133848, abs_tol=1e-6))

Mit 10 Nachbarn ist die Schätzung sehr unruhig, AUC 0,673, mit 200 Nachbarn kommt sie auf 0,725 und bleibt unter der logistischen Regression mit 0,740. Ohne Standardisierung sinkt sie auf 0,711: Dann bestimmen Lagertemperatur und Stromaufnahme mit ihrer großen Streuung in Grad und Prozent, wer Nachbar ist, die Schwingung mit ihrer kleinen Streuung zählt weniger.

## Zum Nachdenken

1. In TODO 1 sagt die Schwingung in Linie 3 weniger voraus als in der Zerspanung. Soll das Werk dort bessere Sensoren einbauen? Was müsstest du dafür noch wissen?
2. Eine Kollegin meldet in TODO 4 stolz eine AUC von 0,90 mit den Störungsmeldungen. Was fragst du zuerst?
3. In TODO 9 hängt die Schwelle an den angenommenen Kosten. Wer im Werk sollte diese Annahmen festlegen, und was tust du, wenn Stillstand an Linie 1 teurer ist als an Linie 4?
4. Nach TODO 12 bekommen die Fräszentren M-01 und M-02 seit dem Tausch in fast jeder zweiten Schicht einen Alarm. Was tust du sofort, und was auf Dauer?
5. Dein Steckbrief aus TODO 13 geht an die Werksleitung. Was fehlt noch, bevor das Modell in Betrieb geht?

<details><summary>Hinweise zu den Antworten</summary>

1. Ein Odds-Verhältnis beschreibt einen Zusammenhang in diesen Daten. Ob andere Sensoren mehr vorhersagen, zeigt nur ein Versuch mit ihnen. Dazu kommen die Kosten der Sensoren und wie teuer Ausfälle in Linie 3 sind; die Linie hat mit 4,7 % eher wenige.
2. Ob jedes Merkmal zum Zeitpunkt der Vorhersage schon bekannt ist. Meldungen der nächsten Schicht gibt es erst nach dem Ereignis, das vorhergesagt werden soll. Eine AUC, die viel besser ist als alles andere, ist ein Warnzeichen.
3. Die Kosten kennen Instandhaltung und Controlling, entscheiden muss die Werksleitung. Unterschiedliche Kosten je Linie führen zu einer eigenen Schwelle je Linie, mit $p^* = C_{FP}/(C_{FP} + C_{FN})$ je Linie, sofern das Modell dort kalibriert ist.
4. Sofort: die Schwelle für Linie 2 anheben oder Alarme dort vorläufig prüfen lassen und den Tausch dokumentieren. Auf Dauer: mit Daten der neuen Sensoren neu trainieren, Merkmale relativ zum Normalwert jeder Maschine bauen und Eingänge, Alarmquote und Kalibrierung je Linie laufend überwachen.
5. Zuständigkeiten, eine Regel, wann neu trainiert wird, die Überwachung, eine Rückmeldung der Inspektionen und eine Erprobung im Betrieb, etwa erst einige Wochen parallel ohne Konsequenzen.
</details>

## Wie es weitergeht

Du hast logistische Regressionen geschätzt und als Odds-Verhältnisse gelesen, eine Zielgröße aus dem Arbeitskalender gebaut, Leckage erzeugt, Konfusionsmatrix, ROC- und Precision-Recall-Kurve gerechnet, eine Schwelle nach Kosten gewählt, die Kalibrierung geprüft und die Güte je Gruppe verglichen. Bisher hatte jedes Merkmal einen festen Koeffizienten. **Modul 9 „Bäume, Ensembles und Feature Engineering“** baut Modelle, die Schwellen und Wechselwirkungen selbst finden, und Merkmale wie den Abstand zum Normalwert jeder Maschine.